# Phase 6 — Feature Exploration Lab: Cross-Asset & Microstructure Expansion

**Goal:** Validate 4 candidate features against the existing 10-dim baseline using the same
KS / MI / temporal profiling methodology as Phase 4 (`phase4_microstructure_exploration.ipynb`).

**Run from:** repo root directory (same level as `src/`, `data/`, `outputs/`, `notebooks/`)

**Candidate features:**
1. `dxy_return_20` — 20-bar DXY log-return z-scored (cross-asset exogenous signal)
2. `vwap_dev_norm` — session-anchored VWAP deviation, ATR-normalised (institutional order flow)
3. `session_enc` — UTC trading session phase, already in NPZ RL obs array (calendar context)
4. `ret_5m / ret_15m / ret_1h` — multi-timeframe returns (explicit multi-scale signal)

**Pass threshold (same as Phase 4):** KS D > 0.05 AND MI > 0.005
**Redundancy threshold (new §6):** max pairwise MI ratio < 0.30

**Outputs → `outputs/`:**
- `phase6_distributions.png`
- `phase6_mutual_information.png`
- `phase6_sequence_profiles_240.png`
- `phase6_divergence_heatmaps.png`
- `phase6_redundancy_heatmap.png`
- `phase6_regime_conditional.png`
- `phase6_recommendation.txt`


## §1 — Setup & Imports

In [ ]:
import sys
import os
import warnings
import numpy as np
import polars as pl
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from scipy import stats
from scipy.stats import ks_2samp
from sklearn.feature_selection import mutual_info_classif
from sklearn.preprocessing import RobustScaler

warnings.filterwarnings('ignore')

# ── Robust repo root resolution (works from any launch directory) ─────────────
# Walks up from this notebook's location until it finds src/ — no manual path config needed.
from pathlib import Path as _Path
_nb_dir  = _Path(os.path.abspath('__file__' if '__file__' in dir() else '.')).resolve()
_repo    = next(
    (p for p in [_nb_dir] + list(_nb_dir.parents) if (p / 'src').is_dir()),
    _nb_dir,
)
REPO_ROOT = str(_repo)
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)
os.chdir(REPO_ROOT)   # make all relative paths (data/, outputs/) work correctly

from src.data.tick_store import TickStore
from src.data.preprocessing import (
    prepare_features, join_regime_labels,
    get_regime_array, compute_rl_obs_features,
)
from src.training.labels import LabelConfig, ATRAdaptiveLabeler

# ── Plot style (match Phase 4) ────────────────────────────────────────────────
plt.style.use('dark_background')
plt.rcParams.update({'figure.dpi': 120, 'font.size': 10})

COLOURS = {'sell': '#E24B4A', 'hold': '#888888', 'buy': '#2ECC71'}
C_DXY   = '#F0C040'   # gold
C_VWAP  = '#AA77FF'   # purple
C_SESS  = '#55CCFF'   # cyan
C_MTF   = '#FF9944'   # orange

print('Setup complete.')
print(f'NumPy {np.__version__} | Polars {pl.__version__}')
print(f'Working directory: {os.getcwd()}')


## §2 — Data Loading

**Fast path (recommended):** if `training_ready.npz` exists locally, set `NPZ_PATH` below.
This is the precomputed cache from supervised training — loads in ~8 s and contains
features, labels, regime arrays, and `session_phase`.

**Full pipeline fallback:** runs `TickStore → prepare_features → join_regime_labels →
ATRAdaptiveLabeler` from the raw `data/ticks.duckdb`. Slower (~5–10 min) but requires
no cached file.

Set `NPZ_PATH = None` to force the full pipeline.


In [3]:
import os, sys, pathlib

# ── Repo root ───────────────────────────────────────────────
REPO_ROOT = pathlib.Path(r"D:\HFTExperiment").resolve()
sys.path.insert(0, str(REPO_ROOT))  # ensure src/ is importable

# ── Imports ─────────────────────────────────────────────────
from src.data.tick_store import TickStore
from src.data.preprocessing import (
    prepare_features,
    join_regime_labels,
    get_regime_array,
    compute_rl_obs_features,
)
from src.training.labels import LabelConfig, ATRAdaptiveLabeler
import numpy as np

# ── Configure paths ───────────────────────────────────────────────────────────
# Point NPZ_PATH to your local training_ready.npz, or set to None for full pipeline
# NPZ is written by: python scripts/precompute_features.py --output data/training_ready.npz
# Adjust the path below if you moved it; set to None to force the full pipeline.
NPZ_PATH   = pathlib.Path(REPO_ROOT) / 'data' / 'training_ready.npz'
DXY_PATH   = pathlib.Path(REPO_ROOT) / 'data' / 'dxy_m1.parquet'   # optional
REGIME_CSV = str(pathlib.Path(REPO_ROOT) / 'data' / 'regime' / 'daily_regime_labels.csv')
WS         = 120  # warmup window, must match training config

os.makedirs('outputs', exist_ok=True)  # ensure outputs dir exists

# ── Load ──────────────────────────────────────────────────────────────────────
if NPZ_PATH is not None and NPZ_PATH.exists():
    print(f'Fast path — loading: {NPZ_PATH}')
    _d = np.load(str(NPZ_PATH), allow_pickle=True)

    features_10d     = _d['features']              # (N, 10) float32
    close_prices     = _d['close']                 # (N,)    float64
    high_prices      = _d.get('high', None)
    low_prices       = _d.get('low',  None)
    labels           = _d['labels']                # (N,)    int64
    gmm2_arr         = _d.get('gmm2',    None)
    vol_arr          = _d.get('vol_enc', None)
    gs_arr           = _d.get('gs_q',    None)
    cu_arr           = _d.get('cu_au',   None)
    rq_arr           = _d.get('rq',      None)
    timestamps_ns    = _d.get('timestamps_ns', None)
    session_phase_arr= _d.get('session_phase', None)
    df_filtered      = None

    print(f'  features: {features_10d.shape}  labels: {labels.shape}')

else:
    print('Full pipeline — loading from ticks.duckdb (~5-10 min)...')
    store  = TickStore(str(pathlib.Path(REPO_ROOT) / 'data' / 'ticks.duckdb'))
    df_raw = store.query_ohlcv('XAUUSD', 'M1')
    store.close()

    df_raw = join_regime_labels(df_raw, REGIME_CSV)

    features_10d, close_prices, high_prices, low_prices = prepare_features(
        df_raw, scaler_method='window_minmax', window_size=WS
    )
    label_cfg = LabelConfig(
        method='atr_adaptive',
        atr_period=14, atr_multiplier_tp=1.5, atr_multiplier_sl=0.75,
        atr_min_tp_pips=150, atr_max_tp_pips=800, max_holding_bars=40,
    )
    labels = ATRAdaptiveLabeler(label_cfg).label(close_prices, high_prices, low_prices)

    # Trim warmup
    features_10d  = features_10d[WS:]
    close_prices  = close_prices[WS:]
    high_prices   = high_prices[WS:]  if high_prices is not None else None
    low_prices    = low_prices[WS:]   if low_prices  is not None else None
    labels        = labels[WS:]

    df_filtered      = df_raw[WS:]
    gmm2_arr         = df_filtered['gmm2_state'].to_numpy()      if 'gmm2_state'       in df_filtered.columns else None
    vol_arr          = df_filtered['vol_regime_enc'].to_numpy()   if 'vol_regime_enc'   in df_filtered.columns else None
    gs_arr           = df_filtered['gs_quartile_enc'].to_numpy()  if 'gs_quartile_enc'  in df_filtered.columns else None
    cu_arr           = df_filtered['cu_au_regime_enc'].to_numpy() if 'cu_au_regime_enc' in df_filtered.columns else None
    rq_arr           = df_filtered['regime_quality_norm'].to_numpy() if 'regime_quality_norm' in df_filtered.columns else None
    timestamps_ns    = None
    session_phase_arr= None

# ── Class masks ───────────────────────────────────────────────────────────────
N = len(labels)
sell_mask = labels == 0
hold_mask = labels == 1
buy_mask  = labels == 2

print(f'\nTotal bars: {N:,}')
print(f'  Sell: {sell_mask.sum():,} ({sell_mask.mean():.2%})')
print(f'  Hold: {hold_mask.sum():,} ({hold_mask.mean():.2%})')
print(f'  Buy:  {buy_mask.sum():,}  ({buy_mask.mean():.2%})')


Fast path — loading: D:\HFTExperiment\data\training_ready.npz
  features: (5680771, 10)  labels: (5680771,)

Total bars: 5,680,771
  Sell: 197,924 (3.48%)
  Hold: 5,439,823 (95.76%)
  Buy:  43,024  (0.76%)


## §3 — Candidate Feature Computation

Each candidate is computed independently from raw arrays and stored as a 1D float32 array
of length N. No modification to `prepare_features()` at this stage — the goal is
validation before any integration decision.


In [4]:
# ── Helper: rolling z-score (chunked, matches preprocessing.py) ──────────────
def rolling_zscore(arr, window=120, epsilon=1e-8):
    arr   = arr.astype(np.float64)
    n     = len(arr)
    out   = np.empty(n, dtype=np.float32)
    CHUNK = 500_000
    for start in range(0, n, CHUNK):
        end      = min(start + CHUNK, n)
        ctx_lo   = max(0, start - (window - 1))
        chunk    = arr[ctx_lo:end]
        nc       = end - start
        pad_rows = window - 1 - (start - ctx_lo)
        if pad_rows > 0:
            chunk = np.concatenate([np.repeat(arr[:1], pad_rows), chunk])
        shape   = (nc, window)
        strides = (chunk.strides[0], chunk.strides[0])
        wins    = np.lib.stride_tricks.as_strided(chunk, shape=shape, strides=strides)
        mu      = wins.mean(axis=1)
        sig     = wins.std(axis=1)
        out[start:end] = np.tanh((arr[start:end] - mu) / (sig + epsilon)).astype(np.float32)
    return out


# ── A: dxy_return_20 ─────────────────────────────────────────────────────────
DXY_AVAILABLE = False
dxy_return_20 = np.zeros(N, dtype=np.float32)

if DXY_PATH.exists():
    try:
        dxy_df    = pl.read_parquet(str(DXY_PATH))
        dxy_close = dxy_df['close'].to_numpy().astype(np.float64)
        dxy_log   = np.concatenate([[0.0], np.log(dxy_close[1:] / (dxy_close[:-1] + 1e-8))])
        dxy_20    = np.convolve(dxy_log, np.ones(20), mode='same')
        if len(dxy_20) >= N:
            dxy_20 = dxy_20[:N]
        else:
            dxy_20 = np.pad(dxy_20, (0, N - len(dxy_20)), mode='edge')
        dxy_return_20 = rolling_zscore(dxy_20, window=120)
        DXY_AVAILABLE = True
        print(f'dxy_return_20  : mean={dxy_return_20.mean():.4f}  std={dxy_return_20.std():.4f}')
    except Exception as e:
        print(f'WARNING: DXY load failed ({e}) — excluded from validation.')
else:
    print(f'NOTE: {DXY_PATH} not found — dxy_return_20 excluded.')
    print('      Place M1 DXY parquet at data/dxy_m1.parquet to enable.')


# ── B: vwap_dev_norm ─────────────────────────────────────────────────────────
def compute_vwap_deviation(close, high, low, tick_vol=None, timestamps_ns=None, atr_period=14):
    n  = len(close)
    tp = (high + low + close) / 3.0
    # ATR (Wilder smoothing)
    tr    = np.maximum(high[1:]-low[1:], np.abs(high[1:]-close[:-1]), np.abs(low[1:]-close[:-1]))
    tr    = np.concatenate([[tr[0]], tr])
    atr   = np.zeros(n, np.float64)
    atr[0]= tr[0]
    a     = 1.0 / atr_period
    for i in range(1, n):
        atr[i] = atr[i-1] * (1 - a) + tr[i] * a
    # Session ID from UTC hour
    if timestamps_ns is not None:
        hours      = (timestamps_ns // (3600 * 1_000_000_000)) % 24
        session_id = np.where(hours < 8, 0, np.where(hours < 16, 1, 2))
    else:
        session_id = (np.arange(n) % 1440) // 480   # synthetic fallback
    # Session-anchored VWAP
    vwap     = np.zeros(n, np.float64)
    cum_pv   = 0.0; cum_vol = 0.0; prev_s = session_id[0]
    for i in range(n):
        if session_id[i] != prev_s:
            cum_pv = 0.0; cum_vol = 0.0; prev_s = session_id[i]
        v = float(tick_vol[i]) if tick_vol is not None else 1.0
        cum_pv  += tp[i] * v;  cum_vol += v
        vwap[i]  = cum_pv / (cum_vol + 1e-8)
    dev = (close - vwap) / (atr + 1e-8)
    return np.tanh(dev).astype(np.float32)

_high     = high_prices if high_prices is not None else close_prices
_low      = low_prices  if low_prices  is not None else close_prices
_tick_vol = None
if df_filtered is not None and 'tick_volume' in df_filtered.columns:
    _tick_vol = df_filtered['tick_volume'].to_numpy().astype(np.float64)

vwap_dev_norm = compute_vwap_deviation(close_prices, _high, _low, _tick_vol, timestamps_ns)
print(f'vwap_dev_norm  : mean={vwap_dev_norm.mean():.4f}  std={vwap_dev_norm.std():.4f}')


# ── C: session_enc ───────────────────────────────────────────────────────────
if session_phase_arr is not None and len(session_phase_arr) == N:
    session_enc = session_phase_arr.astype(np.float32)
    print(f'session_enc    : loaded from NPZ session_phase')
elif timestamps_ns is not None:
    hours       = (timestamps_ns // (3600 * 1_000_000_000)) % 24
    minutes     = (timestamps_ns // (60  * 1_000_000_000)) % 60
    base        = np.where(hours < 8, 0.0, np.where(hours < 16, 0.5, 1.0)).astype(np.float32)
    session_enc = (base + minutes.astype(np.float32) / 60.0 * 0.5 / 8.0).astype(np.float32)
    print(f'session_enc    : computed from timestamps_ns')
else:
    session_enc = ((np.arange(N) % 1440) / 1440.0).astype(np.float32)
    print(f'session_enc    : fallback bar-index cycle (timestamps unavailable)')
print(f'               : mean={session_enc.mean():.4f}  std={session_enc.std():.4f}')


# ── D: multi-timeframe returns ────────────────────────────────────────────────
log_ret = np.concatenate([[0.0], np.log(close_prices[1:] / (close_prices[:-1] + 1e-8))])

def mtf_return(log_ret, window):
    raw = np.convolve(log_ret, np.ones(window, np.float64), mode='same')
    return rolling_zscore(raw, window=120)

ret_5m  = mtf_return(log_ret, 5)
ret_15m = mtf_return(log_ret, 15)
ret_1h  = mtf_return(log_ret, 60)

for name, arr in [('ret_5m', ret_5m), ('ret_15m', ret_15m), ('ret_1h', ret_1h)]:
    print(f'{name:<15}: mean={arr.mean():.4f}  std={arr.std():.4f}')

print('\nAll candidate features computed.')


NOTE: D:\HFTExperiment\data\dxy_m1.parquet not found — dxy_return_20 excluded.
      Place M1 DXY parquet at data/dxy_m1.parquet to enable.
vwap_dev_norm  : mean=0.0286  std=0.9140
session_enc    : loaded from NPZ session_phase
               : mean=0.3493  std=0.3749
ret_5m         : mean=-0.0016  std=0.6068
ret_15m        : mean=-0.0028  std=0.6386
ret_1h         : mean=-0.0043  std=0.7185

All candidate features computed.


## §4 — Discriminative Signal Validation (KS + MI)

In [ ]:
# ── Candidate feature registry ───────────────────────────────────────────────
candidates = {}
if DXY_AVAILABLE:
    candidates['dxy_return_20'] = dxy_return_20
candidates['vwap_dev_norm'] = vwap_dev_norm
candidates['session_enc']   = session_enc
candidates['ret_5m']        = ret_5m
candidates['ret_15m']       = ret_15m
candidates['ret_1h']        = ret_1h

cand_names  = list(candidates.keys())
cand_arrays = np.stack(list(candidates.values()), axis=1)   # (N, n_cands)
n_cands     = len(cand_names)
print(f'Candidate matrix: {cand_arrays.shape}')
print(f'Features: {cand_names}')


In [ ]:
# ── KS tests: sell vs hold, buy vs hold ──────────────────────────────────────
rng = np.random.default_rng(42)
KS_SUB = 10_000

ks_sell, ks_buy = {}, {}
for name in cand_names:
    feat = candidates[name]
    sv   = rng.choice(feat[sell_mask], min(KS_SUB, sell_mask.sum()), replace=False)
    hv   = rng.choice(feat[hold_mask], min(KS_SUB, hold_mask.sum()), replace=False)
    bv   = rng.choice(feat[buy_mask],  min(KS_SUB, buy_mask.sum()),  replace=False)
    d_sh, p_sh = ks_2samp(sv, hv)
    d_bh, p_bh = ks_2samp(bv, hv)
    ks_sell[name] = (d_sh, p_sh)
    ks_buy[name]  = (d_bh, p_bh)

print('KS test — sell vs hold:')
print(f'  {"Feature":<18} {"D":>8} {"p":>12}  Signal')
print('  ' + '-' * 52)
for name, (d, p) in ks_sell.items():
    sig = '✓ STRONG' if d > 0.10 else ('✓ MODERATE' if d > 0.05 else '✗ WEAK')
    print(f'  {name:<18} {d:>8.4f} {p:>12.2e}  {sig}')

print()
print('KS test — buy vs hold:')
print(f'  {"Feature":<18} {"D":>8} {"p":>12}')
print('  ' + '-' * 44)
for name, (d, p) in ks_buy.items():
    print(f'  {name:<18} {d:>8.4f} {p:>12.2e}')


In [ ]:
# ── Distribution plots ───────────────────────────────────────────────────────
n_cols = min(3, n_cands)
n_rows = (n_cands + n_cols - 1) // n_cols
fig, axes = plt.subplots(n_rows, n_cols, figsize=(6 * n_cols, 4.5 * n_rows))
axes = np.array(axes).flatten()
N_HIST = min(50_000, sell_mask.sum())

for i, name in enumerate(cand_names):
    ax   = axes[i]
    feat = candidates[name]
    for label, mask, colour in [('sell', sell_mask, COLOURS['sell']),
                                  ('hold', hold_mask, COLOURS['hold']),
                                  ('buy',  buy_mask,  COLOURS['buy'])]:
        idx  = np.where(mask)[0]
        samp = rng.choice(idx, min(N_HIST, len(idx)), replace=False)
        vals = feat[samp]
        lo, hi = np.percentile(vals, 1), np.percentile(vals, 99)
        ax.hist(np.clip(vals, lo, hi), bins=80, density=True,
                alpha=0.45, color=colour, label=label)
    d, p = ks_sell[name]
    ax.set_title(f'{name}\nKS(sell vs hold) D={d:.3f}  p={p:.2e}', fontsize=9)
    ax.legend(fontsize=8)
    ax.grid(alpha=0.2)

for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

fig.suptitle('Phase 6 — Candidate Feature Distributions', fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig('outputs/phase6_distributions.png', bbox_inches='tight', dpi=150)
plt.show()
print('Saved: outputs/phase6_distributions.png')


In [ ]:
# ── Mutual information: candidates + existing 10D baseline ───────────────────
BASELINE_NAMES = [
    'open_sc', 'high_sc', 'low_sc', 'close_sc', 'tickvol_sc', 'spread_sc',
    'bar_return_bps', 'wick_asymmetry', 'vol_zscore', 'spread_pressure',
]
scaler_cand  = RobustScaler()
cand_scaled  = scaler_cand.fit_transform(cand_arrays)
combined_mat = np.concatenate([features_10d, cand_scaled], axis=1)
combined_names = BASELINE_NAMES + cand_names

rng_mi = np.random.default_rng(42)
n_mi   = min(60_000, N)
idx_mi = rng_mi.choice(N, n_mi, replace=False)

print('Computing mutual information...')
mi_scores = mutual_info_classif(
    combined_mat[idx_mi], labels[idx_mi],
    discrete_features=False, random_state=42,
)
mi_baseline  = mi_scores[:10]
mi_candidate = mi_scores[10:]

print('\nMI — existing 10D baseline:')
for name, mi in zip(BASELINE_NAMES, mi_baseline):
    print(f'  {name:<22}: {mi:.5f}')

print('\nMI — candidate features:')
PASS_MI = 0.005
for name, mi in zip(cand_names, mi_candidate):
    tag = '✓ PASS' if mi > PASS_MI else '✗ FAIL'
    print(f'  {name:<22}: {mi:.5f}  {tag}')

print(f'\nBaseline OHLCV mean:         {mi_baseline[:6].mean():.5f}')
print(f'Baseline microstructure mean: {mi_baseline[6:].mean():.5f}')
print(f'Candidate mean:               {mi_candidate.mean():.5f}')

# ── Plot ──────────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(14, 6))
bar_colours = (
    ['#5588CC'] * 6 + ['#E24B4A'] * 4 +
    ([C_DXY] if DXY_AVAILABLE else []) +
    [C_VWAP, C_SESS, C_MTF, C_MTF, C_MTF]
)[:len(combined_names)]

ax.barh(combined_names[::-1], mi_scores[::-1], color=bar_colours[::-1], alpha=0.85)
ax.axvline(mi_baseline[:6].mean(),  color='#5588CC', ls='--', lw=1.2, label='OHLCV mean MI')
ax.axvline(mi_baseline[6:].mean(),  color='#E24B4A', ls='--', lw=1.2, label='Micro mean MI')
ax.axvline(PASS_MI, color='white',  ls=':',  lw=1.0, label=f'Pass threshold {PASS_MI}')
ax.set_xlabel('Mutual Information with label')
ax.set_title('Phase 6 — MI Ranking: 10D Baseline vs Candidate Features')
ax.legend(fontsize=9)
ax.grid(alpha=0.2, axis='x')
plt.tight_layout()
plt.savefig('outputs/phase6_mutual_information.png', bbox_inches='tight', dpi=150)
plt.show()
print('Saved: outputs/phase6_mutual_information.png')


## §5 — Temporal Profiling at 240-bar Depth

For each candidate: mean ± SE sequence profiles + per-bar Cohen's d / KS stat.

**Key question:** Is divergence concentrated in the last 20 bars (short stream, `LocalCausalAttention w=20`)
or distributed across 240 bars (long stream, Transformer)? This determines injection point if the feature passes.


In [ ]:
SEQ_LEN_EXT  = 240
N_SAMPLE_SEQ = 2000
rng_seq = np.random.default_rng(42)

sell_idx = rng_seq.choice(np.where(sell_mask)[0], min(N_SAMPLE_SEQ, sell_mask.sum()), replace=False)
hold_idx = rng_seq.choice(np.where(hold_mask)[0], min(N_SAMPLE_SEQ, hold_mask.sum()), replace=False)
buy_idx  = rng_seq.choice(np.where(buy_mask)[0],  min(N_SAMPLE_SEQ, buy_mask.sum()),  replace=False)

def collect_traces(feat_arr, idx_set, seq_len):
    traces = []
    for seq_end in idx_set:
        start = max(0, seq_end - seq_len + 1)
        seq   = feat_arr[start:seq_end + 1]
        if len(seq) == seq_len:
            traces.append(seq)
    return np.array(traces)

# ── Mean ± SE profiles ────────────────────────────────────────────────────────
n_cols = min(3, n_cands)
n_rows = (n_cands + n_cols - 1) // n_cols
fig, axes = plt.subplots(n_rows, n_cols, figsize=(7 * n_cols, 4.5 * n_rows))
axes = np.array(axes).flatten()
t    = np.arange(SEQ_LEN_EXT)

for i, name in enumerate(cand_names):
    ax = axes[i]
    for label, idx_set, colour in [('sell', sell_idx, COLOURS['sell']),
                                    ('hold', hold_idx, COLOURS['hold']),
                                    ('buy',  buy_idx,  COLOURS['buy'])]:
        tr = collect_traces(candidates[name], idx_set, SEQ_LEN_EXT)
        if len(tr) == 0: continue
        mean = tr.mean(0);  se = tr.std(0) / np.sqrt(len(tr))
        ax.fill_between(t, mean - se, mean + se, alpha=0.25, color=colour)
        ax.plot(t, mean, color=colour, lw=1.5, label=label)
    ax.axvline(SEQ_LEN_EXT - 20, color='white',   lw=0.8, ls='--', alpha=0.6, label='last 20 bars')
    ax.axvline(SEQ_LEN_EXT - 60, color='#AAAAAA', lw=0.6, ls=':',  alpha=0.4, label='last 60 bars')
    ax.set_title(f'{name} (240-bar)', fontsize=9)
    ax.legend(fontsize=7)
    ax.set_xlabel('bar position in sequence')
    ax.grid(alpha=0.2)

for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

fig.suptitle('Phase 6 — 240-bar Sequence Profiles: Sell vs Hold vs Buy (±1 SE)', fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig('outputs/phase6_sequence_profiles_240.png', bbox_inches='tight', dpi=150)
plt.show()
print('Saved: outputs/phase6_sequence_profiles_240.png')


In [ ]:
# ── Per-bar Cohen's d + KS — divergence heatmaps ─────────────────────────────
divergence_results = {}

for name in cand_names:
    feat    = candidates[name]
    sell_tr = collect_traces(feat, sell_idx, SEQ_LEN_EXT)
    hold_tr = collect_traces(feat, hold_idx, SEQ_LEN_EXT)
    buy_tr  = collect_traces(feat, buy_idx,  SEQ_LEN_EXT)

    def per_bar(a_arr, b_arr):
        d_vals, ks_vals = [], []
        for t in range(SEQ_LEN_EXT):
            a, b = a_arr[:, t], b_arr[:, t]
            ps   = np.sqrt(((a.std()**2) + (b.std()**2)) / 2)
            d_vals.append((a.mean() - b.mean()) / (ps + 1e-8))
            ks_vals.append(ks_2samp(a, b)[0])
        return np.array(d_vals), np.array(ks_vals)

    d_sh, ks_sh = per_bar(sell_tr, hold_tr)
    d_bh, ks_bh = per_bar(buy_tr,  hold_tr)
    divergence_results[name] = {
        'sell_vs_hold': {'cohen_d': d_sh, 'ks_stat': ks_sh},
        'buy_vs_hold':  {'cohen_d': d_bh, 'ks_stat': ks_bh},
    }

cohen_sell = np.array([divergence_results[n]['sell_vs_hold']['cohen_d'] for n in cand_names])
cohen_buy  = np.array([divergence_results[n]['buy_vs_hold']['cohen_d']  for n in cand_names])

fig, axes = plt.subplots(1, 2, figsize=(16, max(3, n_cands * 0.9 + 1)))
for ax, mat, title in [(axes[0], cohen_sell, "Cohen's d — Sell vs Hold"),
                        (axes[1], cohen_buy,  "Cohen's d — Buy vs Hold")]:
    im = ax.imshow(mat, aspect='auto', cmap='RdBu_r',
                   interpolation='nearest', vmin=-0.8, vmax=0.8)
    ax.set_title(title, fontsize=10)
    ax.set_yticks(np.arange(n_cands)); ax.set_yticklabels(cand_names, fontsize=8)
    ax.set_xlabel('Bar position (0–240)')
    ax.axvline(SEQ_LEN_EXT - 20, color='white',   lw=1,   ls='--', alpha=0.8)
    ax.axvline(SEQ_LEN_EXT - 60, color='#AAAAAA', lw=0.7, ls=':',  alpha=0.5)

fig.colorbar(im, ax=axes.ravel().tolist(), shrink=0.6, label="Effect size (Cohen's d)")
fig.suptitle('Phase 6 — Per-bar Divergence Heatmaps (240-bar horizon)', fontsize=13)
plt.tight_layout()
plt.savefig('outputs/phase6_divergence_heatmaps.png', bbox_inches='tight', dpi=150)
plt.show()
print('Saved: outputs/phase6_divergence_heatmaps.png')


In [ ]:
# ── Divergence summary table + stream assignment ─────────────────────────────
print(f'{"Feature":<18}  {"SellD[220]":>10} {"SellD[20]":>10} {"SellKS[20]":>11}  '
      f'{"BuyD[220]":>10} {"BuyD[20]":>10}  {"Stream":>12}')
print('-' * 90)

stream_assignment = {}
for name in cand_names:
    m  = divergence_results[name]
    ds = m['sell_vs_hold']['cohen_d']
    ks = m['sell_vs_hold']['ks_stat']
    db = m['buy_vs_hold']['cohen_d']

    d_se  = np.abs(ds[:-20]).mean(); d_sl = np.abs(ds[-20:]).mean()
    d_be  = np.abs(db[:-20]).mean(); d_bl = np.abs(db[-20:]).mean()
    ks_sl = ks[-20:].mean()
    ratio = d_sl / (d_se + 1e-6)
    stream = 'short (w=20)' if ratio > 1.5 else ('both' if ratio > 0.8 else 'long')
    stream_assignment[name] = stream

    print(f'{name:<18}  {d_se:>10.4f} {d_sl:>10.4f} {ks_sl:>11.4f}  '
          f'{d_be:>10.4f} {d_bl:>10.4f}  {stream:>12}')

print()
print('Stream guide: short→LocalCausalAttention | long→Transformer | both→post-fusion inject')


## §6 — Redundancy Analysis

Tests whether each candidate adds information the existing 10D features do not already carry.
A feature with high pairwise MI against an existing feature will produce collinear gradients
in the model — the Transformer already learns their joint representation.

**Verdict threshold:** `max(MI[candidate, existing_i]) / MI[candidate, label] < 0.30`


In [ ]:
# ── Pairwise MI: each candidate vs each existing feature ─────────────────────
pairwise_mi = np.zeros((n_cands, 10), dtype=np.float32)

for ci, name in enumerate(cand_names):
    cand_col = cand_scaled[idx_mi, ci]
    cand_bin = (cand_col > np.median(cand_col)).astype(int)
    for bi in range(10):
        base_col = features_10d[idx_mi, bi].reshape(-1, 1)
        pairwise_mi[ci, bi] = mutual_info_classif(
            base_col, cand_bin,
            discrete_features=False, random_state=42,
        )[0]

# ── Heatmap ───────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(13, max(3, n_cands * 0.9 + 1.5)))
im = ax.imshow(pairwise_mi, aspect='auto', cmap='YlOrRd', vmin=0)
ax.set_xticks(np.arange(10))
ax.set_xticklabels(BASELINE_NAMES, rotation=45, ha='right', fontsize=8)
ax.set_yticks(np.arange(n_cands))
ax.set_yticklabels(cand_names, fontsize=8)
ax.set_title('Phase 6 — Pairwise MI: Candidates × Existing 10D Features', fontsize=11)
plt.colorbar(im, ax=ax, label='Mutual Information')

for ci in range(n_cands):
    for bi in range(10):
        v = pairwise_mi[ci, bi]
        ax.text(bi, ci, f'{v:.3f}', ha='center', va='center',
                fontsize=6, color='black' if v > 0.03 else 'white')

plt.tight_layout()
plt.savefig('outputs/phase6_redundancy_heatmap.png', bbox_inches='tight', dpi=150)
plt.show()
print('Saved: outputs/phase6_redundancy_heatmap.png')

# ── Verdict ───────────────────────────────────────────────────────────────────
print()
print(f'{"Feature":<18}  {"MaxPairMI":>10} {"LabelMI":>10} {"Ratio":>8}  Verdict')
print('-' * 65)
PASS_RED = 0.30
redundancy_verdict = {}
for ci, name in enumerate(cand_names):
    max_pw = pairwise_mi[ci].max()
    lab_mi = mi_candidate[ci]
    ratio  = max_pw / (lab_mi + 1e-8)
    verdict = '✓ ADDITIVE' if ratio < PASS_RED else '✗ REDUNDANT'
    redundancy_verdict[name] = ratio < PASS_RED
    print(f'{name:<18}  {max_pw:>10.4f} {lab_mi:>10.4f} {ratio:>8.3f}  {verdict}')


## §7 — Regime-Conditional Feature Behaviour

Splits by GMM2 (Bear/Bull). Key questions:
- Is `dxy_return_20` more discriminative in Bear+HIGH (risk-off, tight gold/dollar correlation)?
- Does `vwap_dev_norm` separate sell better in Bull (trending) than Bear (mean-reverting)?


In [ ]:
if gmm2_arr is not None and len(gmm2_arr) == N:
    bear_r = gmm2_arr < 0.5
    bull_r = gmm2_arr >= 0.5

    n_cols = min(3, n_cands)
    n_rows = (n_cands + n_cols - 1) // n_cols
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(6 * n_cols, 4.5 * n_rows))
    axes = np.array(axes).flatten()
    regime_ks = {}

    for i, name in enumerate(cand_names):
        ax   = axes[i]
        feat = candidates[name]
        lo, hi = np.percentile(feat, 2), np.percentile(feat, 98)
        fc     = np.clip(feat, lo, hi)

        for label, mask, colour, lname in [
            (None, bear_r & sell_mask, '#E24B4A', 'Bear+sell'),
            (None, bull_r & sell_mask, '#F09020', 'Bull+sell'),
            (None, bear_r & hold_mask, '#5599CC', 'Bear+hold'),
        ]:
            if mask.sum() > 10:
                ax.hist(fc[mask], bins=60, density=True, alpha=0.5, color=colour, label=lname)

        bs  = feat[bear_r & sell_mask]
        bus = feat[bull_r & sell_mask]
        if len(bs) > 30 and len(bus) > 30:
            ks_rb, _ = ks_2samp(
                rng.choice(bs,  min(5000, len(bs)),  replace=False),
                rng.choice(bus, min(5000, len(bus)), replace=False),
            )
            regime_ks[name] = ks_rb
            ax.set_title(f'{name}\nKS(Bear+sell vs Bull+sell) D={ks_rb:.3f}', fontsize=8)
        else:
            ax.set_title(f'{name}\n(insufficient Bear+sell samples)', fontsize=8)

        ax.legend(fontsize=7); ax.grid(alpha=0.2)

    for j in range(i + 1, len(axes)):
        axes[j].set_visible(False)

    plt.suptitle('Phase 6 — Regime-Conditional: Bear vs Bull × Sell vs Hold', fontsize=12, y=1.01)
    plt.tight_layout()
    plt.savefig('outputs/phase6_regime_conditional.png', bbox_inches='tight', dpi=150)
    plt.show()
    print('Saved: outputs/phase6_regime_conditional.png')

    print('\nRegime sensitivity:')
    for name, ks in regime_ks.items():
        tag = 'regime-sensitive' if ks > 0.10 else 'regime-stable'
        print(f'  {name:<18}: D={ks:.4f}  ({tag})')
else:
    print('Regime arrays not available — skipping (re-run full pipeline to enable).')
    regime_ks = {}


## §8 — Recommendation Summary

3-criterion automated verdict: **KS** (D > 0.05) · **MI** (> 0.005) · **Redundancy** (ratio < 0.30)

All 3 must pass → supervised branch inclusion.
KS + MI pass but redundancy fails → RL obs space only.
KS or MI fail → exclude.


In [ ]:
PASS_KS  = 0.05
PASS_MI  = 0.005
PASS_RED = 0.30

print('=' * 72)
print('PHASE 6 — FEATURE VALIDATION REPORT')
print(f'Thresholds: KS > {PASS_KS} | MI > {PASS_MI} | Redundancy ratio < {PASS_RED}')
print('=' * 72)

recommendations = {}
report_lines    = []

for ci, name in enumerate(cand_names):
    ks_d      = ks_sell[name][0]
    lab_mi    = mi_candidate[ci]
    max_pw    = pairwise_mi[ci].max()
    red_ratio = max_pw / (lab_mi + 1e-8)
    stream    = stream_assignment.get(name, '?')

    p_ks  = ks_d    > PASS_KS
    p_mi  = lab_mi  > PASS_MI
    p_red = red_ratio < PASS_RED

    if p_ks and p_mi and p_red:
        rec = 'INCLUDE — supervised branch'
    elif p_ks and p_mi and not p_red:
        rec = 'INCLUDE — RL obs space only (redundant with existing 10D)'
    elif not (p_ks or p_mi):
        rec = 'EXCLUDE'
    else:
        rec = 'MARGINAL — manual review'

    recommendations[name] = rec
    line = (
        f'{name:<20}  KS={ks_d:.3f}{"✓" if p_ks else "✗"}  '
        f'MI={lab_mi:.4f}{"✓" if p_mi else "✗"}  '
        f'Red={red_ratio:.2f}{"✓" if p_red else "✗"}  '
        f'{stream:<14}→ {rec}'
    )
    report_lines.append(line)
    print(line)

sup_pass = [n for n, r in recommendations.items() if 'supervised' in r]
rl_only  = [n for n, r in recommendations.items() if 'RL obs' in r]
excluded = [n for n, r in recommendations.items() if r == 'EXCLUDE']

print()
print('─' * 72)
print(f'Supervised additions ({len(sup_pass)}): {sup_pass}')
if sup_pass:
    print(f'  → input_dim: 10 → {10 + len(sup_pass)}')
    print(f'  → Rebuild NPZ · scattering bypass · Run 8')
print(f'RL obs additions  ({len(rl_only)}): {rl_only}')
print(f'Excluded          ({len(excluded)}): {excluded}')

# ── Save report ───────────────────────────────────────────────────────────────
import datetime
txt = '\n'.join([
    f'Phase 6 Feature Validation — {datetime.date.today()}',
    f'Thresholds: KS > {PASS_KS} | MI > {PASS_MI} | Redundancy ratio < {PASS_RED}',
    '=' * 72,
    '',
] + report_lines + [
    '',
    f'Supervised: {sup_pass}  (input_dim 10 → {10 + len(sup_pass)})',
    f'RL obs:     {rl_only}',
    f'Excluded:   {excluded}',
])
with open('outputs/phase6_recommendation.txt', 'w') as f:
    f.write(txt)
print('\nSaved: outputs/phase6_recommendation.txt')
print('=' * 72)


## §9 — NPZ Rebuild & Run 8 Spec

Conditional: only executes if features passed for the supervised branch.
Prints the exact `preprocessing.py` change, scattering bypass architecture,
config delta, and Supervised Run 8 command.


In [ ]:
sup_pass = [n for n, r in recommendations.items() if 'supervised' in r]

if not sup_pass:
    print('No supervised-pass features — no NPZ rebuild required.')
    print('Proceed to RL training with current training_ready.npz unchanged.')
else:
    new_dim  = 10 + len(sup_pass)
    n_bypass = len(sup_pass)
    tag      = f'phase6_{n_bypass}feat'

    print(f'NPZ rebuild required: {n_bypass} new feature(s), input_dim 10 → {new_dim}')
    print()

    print('── src/data/preprocessing.py ── append in prepare_features() after scaled_10d:')
    for name in sup_pass:
        if name == 'dxy_return_20':
            print('    dxy_z     = compute_dxy_return_20(dxy_df, window=120)')
        elif name == 'vwap_dev_norm':
            print('    vwap_dev  = compute_vwap_deviation(close, high, low, tick_vol, timestamps_ns)')
        elif name == 'session_enc':
            print('    sess_enc  = compute_session_enc(timestamps_ns)')
        elif name in ('ret_5m', 'ret_15m', 'ret_1h'):
            w = {'ret_5m': 5, 'ret_15m': 15, 'ret_1h': 60}[name]
            print(f'    {name:<10}= mtf_return(log_ret, window={w})')

    print(f'    extra     = np.stack([{", ".join(sup_pass)}], axis=1)')
    print(f'    extra_sc  = RobustScaler().fit_transform(extra)')
    print(f'    scaled_{new_dim}d = np.concatenate([scaled_10d, extra_sc], axis=1)')
    print()

    print('── src/encoder/price_branch_transformer.py — scattering bypass:')
    print('   Scattering input stays 10D (OHLCV+micro unchanged).')
    print(f'   Add Linear({n_bypass}, d_model) bypass; inject at post-scatter_proj concat:')
    print()
    print('   # __init__:')
    print(f'   self.bypass_proj = nn.Linear({n_bypass}, self.d_model)  # new')
    print()
    print('   # forward(self, x):  x shape (B, T, 10+n_bypass)')
    print(f'   x_base   = x[:, :, :10]          # original 10D → scattering')
    print(f'   x_bypass = x[:, -1, 10:]          # last bar of {n_bypass} new dims')
    print(f'   bypass   = self.bypass_proj(x_bypass)  # (B, d_model)')
    print(f'   # after attention_pool: long_pooled = long_pooled + bypass')
    print()

    print('── configs/model/dual_branch.yaml:')
    print(f'   # input_dim stays 10 (scattering unchanged)')
    print(f'   n_bypass_features: {n_bypass}')
    print()

    print('── Cache-bust: rename NPZ to prevent stale loads:')
    print(f'   training_ready_{tag}.npz')
    print()

    print('── Supervised Run 8:')
    print(f'   python scripts/train_supervised.py \\')
    print(f'       data.npz_path=training_ready_{tag}.npz \\')
    print(f'       model.n_bypass_features={n_bypass} \\')
    print(f'       training.label_smoothing=0.1 \\')
    print(f'       training.resume_from=null \\')
    print(f'       training.epochs=90 \\')
    print(f'       seed=42')


## §10 — Checklist

| Criterion | Threshold | Cell |
|-----------|-----------|------|
| KS(sell vs hold) | D > 0.05, ≥ 2 candidates | §4 |
| Mutual information | MI > 0.005, candidate > baseline micro mean | §4 |
| Redundancy | max pairwise MI ratio < 0.30 | §6 |
| Temporal locality | divergence in identifiable window (stream assignment) | §5 |
| Regime sensitivity | KS(Bear+sell vs Bull+sell) | §7 |

**≥ 2 features pass all 3 criteria →** rebuild NPZ per §9, Run 8.
**0–1 features pass →** precision ceiling is in the backbone. Proceed to RL; revisit features post-RL.

Reference: `PATCH_NOTES_PHASE6_03052026.md` §4 for full theoretical justification.
Phase 4 baseline KS values: `bar_return_bps`=0.162, `wick_asymmetry`=0.252, `vol_zscore`=0.111, `spread_pressure`=0.600.
